# MANATUABON PHASE 9: KAGGLE LORA FINE-TUNING

## NVIDIA Nemotron-3-Nano-30B Reasoning Optimization
Run this notebook on **Google Colab Pro** with an **A100 or L4 GPU**.
Make sure to upload `synthetic_cot.jsonl` (generated by `kaggle_pipeline.py`) to the Colab workspace before running.

In [ ]:
# 1. Install Required Dependencies
!pip install -U transformers peft accelerate bitsandbytes datasets trl

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

def format_prompt(sample):
    """
    Format the ShareGPT conversational format generated by kaggle_pipeline.py 
    into a continuous prompt string for Next-Token Prediction.
    """
    convo = sample["conversations"]
    prompt = convo[0]["value"]
    response = convo[1]["value"]
    
    # Simple template: User prompt + Assistant reasoning and \boxed{} answer
    text = f"User: {prompt}\n\nAssistant: {response}"
    return {"text": text}

# 2. Load the generated dataset (Upload synthetic_cot.jsonl to your Colab workspace)
dataset_path = "synthetic_cot.jsonl"
if not os.path.exists(dataset_path):
    print(f"\n❌ ERROR: Please upload {dataset_path} to your Colab workspace!")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    dataset = dataset.map(format_prompt)
    
    # Create a 90/10 Train/Eval split for Early Stopping
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_data = dataset_split["train"]
    eval_data = dataset_split["test"]
    print(f"\n✅ Loaded {len(train_data)} training and {len(eval_data)} validation traces.")

In [ ]:
# 3. Configure 4-bit Quantization to fit 30B in VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 4. Load the Nemotron 30B Nano Model & Tokenizer
model_id = "nvidia/nemotron-3-nano-30b-instruct"  # Or the specific Kaggle competition unsloth model ID
print(f"Loading base model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

In [ ]:
# 5. Configure the LoRA Adapter (Rank 32 maximum per Kaggle rules)
print("Configuring Rank-32 LoRA adapter...")
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 6. Set up the SFTTrainer with Early Stopping
output_dir = "./nemotron_reasoning_lora"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=150, # Set a high limit, early stopping will halt it sooner if needed
    learning_rate=2e-4,
    fp16=True, # A100 supports bf16, T4 uses fp16
    logging_steps=5,
    eval_strategy="steps", # Required for early stopping
    eval_steps=10,         # Check validation loss every 10 steps
    save_strategy="steps",
    save_steps=10,
    load_best_model_at_end=True, # Loads the model from the step with highest eval score
    metric_for_best_model="eval_loss",
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=4096, # High sequence length required for <think> CoT traces
    tokenizer=tokenizer,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Halts if eval loss goes up 3 times in a row
)

# 7. Train the model
print("\n🔥 Starting LoRA Fine-tuning with Early Stopping...")
trainer.train()

In [ ]:
# 8. Save the final adapter for Kaggle submission
print(f"\n✅ Training complete. Saving LoRA adapter to {output_dir}")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Adapter is ready to be zipped and submitted to Kaggle!")